# Fine-tune Qwen2.5-3B-Instruct for Astrologer Chatbot

QLoRA SFT on Google Colab (T4 GPU).

**Before running:**
1. Upload your `astrology-llm/` project folder to Google Drive
2. Mount Drive and set `PROJECT_PATH` below
3. Runtime → Change runtime type → T4 GPU

In [ ]:
# @title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# @param {type:'string'}
PROJECT_PATH = '/content/drive/MyDrive/astrology-llm'  # @param {type:'string'}

import sys
sys.path.insert(0, PROJECT_PATH)
%cd $PROJECT_PATH

In [ ]:
# @title Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers>=4.40 accelerate>=0.28 peft>=0.8 bitsandbytes>=0.43 trl>=0.8
!pip install -q datasets tensorboard scikit-learn nltk rouge-score
!pip install -q pyyaml pydantic rich

In [ ]:
# @title Verify GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ERROR: No GPU detected. Change runtime to T4 GPU.')

In [ ]:
# @title Run data pipeline (skip if already processed locally)
!python run_pipeline.py --input data/raw/chat_data.json --output-dir data/processed

In [ ]:
# @title Start training
# Training takes ~2-3 hours on T4
# Monitor in TensorBoard at the link below

%load_ext tensorboard
%tensorboard --logdir training/logs

!python -m src.training.train --config configs/training.yaml

In [ ]:
# @title Merge LoRA adapters into base model
!python -m src.training.merge \
    --adapter-path training/checkpoints/final \
    --output-path training/merged/qwen2.5-3b-astrologer

In [ ]:
# @title Test inference with merged model
from src.inference.chat import Conversation
from src.inference.engine import InferenceEngine

engine = InferenceEngine(model_path='training/merged/qwen2.5-3b-astrologer')
chat = Conversation(system_prompt='You are Vedaz AI astrologer.')

chat.add_user('Mera naam Rahul hai. DOB 15 March 1995, 8:30 AM, Delhi.')
response = engine.generate(chat.messages)
print(f'Assistant: {response}')

In [ ]:
# @title Save checkpoint to Drive
import shutil
from pathlib import Path

drive_checkpoint = Path(PROJECT_PATH) / 'training' / 'checkpoints'
local_checkpoint = Path('training/checkpoints')
if local_checkpoint.exists():
    shutil.copytree(local_checkpoint, drive_checkpoint, dirs_exist_ok=True)
    print(f'Checkpoints saved to {drive_checkpoint}')
else:
    print('No checkpoints found')